# Agentic Demand Forecasting & Forecast Exception Management — Colab runner
**Pipeline:** Demand Data → Forecast → Detect Exception → Investigate → Reforecast → Recommend → Human Approval

Four CrewAI agents (Demand Analyst, Forecasting, Forecast Diagnosis, Planning Recommendation) on Gemini, with seven deterministic tools.
All data is synthetic (fictional *Kaveri Foods*). Runtime: Python 3.10–3.13.

**Steps:** 1) get the code · 2) install · 3) add your Gemini key · 4) run rules baseline (no key) · 5) run agents · 6) approve · 7) evaluate.

### 1 · Get the code
Either clone the repo (set `REPO_URL`) or upload `demand-exception-agents.zip` when prompted.

In [ ]:
import os, zipfile
REPO_URL = ""   # e.g. "https://github.com/<team>/demand-exception-agents.git"
if not os.path.exists("demand-exception-agents"):
    if REPO_URL:
        !git clone -q {REPO_URL} demand-exception-agents
    else:
        from google.colab import files
        up = files.upload()                      # choose demand-exception-agents.zip
        zipfile.ZipFile(next(iter(up))).extractall(".")
%cd demand-exception-agents
!ls

### 2 · Install (about 1–2 minutes)

In [ ]:
!pip install -q -r requirements.txt

### 3 · Gemini API key
Free key from https://aistudio.google.com. In Colab, add it under 🔑 *Secrets* as `GEMINI_API_KEY`, or paste when prompted.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key: ")
os.environ["GEMINI_MODEL"] = "gemini-3.8-flash"   # Google retired 2.5-flash for new keys; use any model your key lists in AI Studio

### 4 · Rule-based baseline (no LLM, runs in a second)
Same tools and triage, fixed if-then rules. This is the comparison point.

In [ ]:
!python run.py --mode rules

### 5 · The 4-agent crew
Verbose output shows each agent's tool calls. Expect 2–5 minutes on the free tier. `--approve ask` lets you approve / reject / edit each recommendation in the cell — that is the human-approval step.

In [ ]:
!python run.py --mode agents --approve ask

### 6 · Look at one IWL: history, baseline, proposal and the hidden future

In [ ]:
import sys, json, glob
sys.path.insert(0, "src")
import pandas as pd, matplotlib.pyplot as plt
from engine import get_engine, N_HIST
e = get_engine()
run = json.load(open(sorted(glob.glob("runs/agents_*.json"))[-1]))
fut = pd.read_csv("data/future_actuals.csv")
iwl = "POP150-DEL"     # try OAT1K-MUM, GRA12-MUM, POP150-MUM, OAT1K-KOL
b = e.baseline[iwl]; w = list(range(N_HIST-51, N_HIST+1)); fw = list(range(N_HIST+1, N_HIST+13))
plt.figure(figsize=(11,4))
plt.plot(w, e.Y[iwl][-52:], label="actual"); plt.plot(fw, b["fcst"], label="baseline forecast")
if iwl in run.get("proposals", {}): plt.plot(fw, run["proposals"][iwl]["reforecast"], label="agent proposal")
plt.plot(fw, fut[fut.iwl_id==iwl].sort_values("week").units, "k--", label="hidden actual (eval only)")
plt.axvline(N_HIST+.5, color="grey"); plt.title(f"{iwl} — {e.label(iwl)}"); plt.legend(); plt.show()

### 7 · Evaluation of a saved run (agents vs rules)

In [ ]:
for f in sorted(glob.glob("runs/*.json"))[-2:]:
    print("\n\n##########", f)
    !python run.py --report {f}

### 8 · Optional: launch the Streamlit app from Colab
The deployed version runs on Streamlit Community Cloud; this is a fallback.

In [ ]:
!npm install -q localtunnel >/dev/null 2>&1
!streamlit run streamlit_app.py --server.port 8501 >/dev/null 2>&1 &
!sleep 5 && curl -s ipv4.icanhazip.com   # this IP is the tunnel password
!npx localtunnel --port 8501